In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
featured_df = pd.read_csv("../data/ndvi_features.csv")

print("Dataset shape:", featured_df.shape)
print(featured_df.head())

Dataset shape: (908, 17)
         lon       lat       label  ndvi_mean  ndvi_std  ndvi_min  ndvi_max  \
0 -54.595477 -3.641303  deforested   0.599597  0.150174  0.337451  0.870792   
1 -54.571395 -3.615745  deforested   0.506074  0.177897  0.303279  0.848954   
2 -54.558029 -3.536081  deforested   0.495752  0.233901  0.146978  0.878971   
3 -54.651907 -3.637025  deforested   0.600711  0.158853  0.395559  0.869378   
4 -54.656760 -3.634666  deforested   0.579386  0.199627  0.314171  0.889375   

   ndvi_range  ndvi_first  ndvi_last  overall_change  largest_drop  \
0    0.533341    0.870792   0.552053       -0.318739     -0.091442   
1    0.545676    0.844116   0.506840       -0.337276     -0.147837   
2    0.731993    0.867034   0.568730       -0.298304     -0.356110   
3    0.473819    0.867872   0.416116       -0.451756     -0.114539   
4    0.575204    0.889375   0.421703       -0.467672     -0.106523   

   ndvi_slope  early_mean  late_mean  early_late_change  min_ndvi_timestep  
0 

In [3]:
feature_cols = [
    "ndvi_mean",
    "ndvi_std",
    "ndvi_min",
    "ndvi_max",
    "ndvi_range",
    "ndvi_first",
    "ndvi_last",
    "overall_change",
    "largest_drop",
    "ndvi_slope",
    "early_mean",
    "late_mean",
    "early_late_change",
    "min_ndvi_timestep"
]

X = featured_df[feature_cols]

y = featured_df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

X shape: (908, 14)
y shape: (908,)

Class distribution:
label
forest          500
old_clearing    292
deforested      116
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (726, 14)
X_test: (182, 14)
y_train: (726,)
y_test: (182,)


In [5]:
def evaluate_model(name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="macro"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall: {recall:.4f}")
    print(f"Macro F1: {f1:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred
        )
    )

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1
    }

In [6]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(X_test)

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    rf_pred
)


Random Forest
Accuracy: 0.7363
Macro Precision: 0.7051
Macro Recall: 0.6709
Macro F1: 0.6853

Classification Report:
              precision    recall  f1-score   support

  deforested       0.63      0.52      0.57        23
      forest       0.76      0.83      0.79       100
old_clearing       0.72      0.66      0.69        59

    accuracy                           0.74       182
   macro avg       0.71      0.67      0.69       182
weighted avg       0.73      0.74      0.73       182



In [7]:
et_model = ExtraTreesClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

et_model.fit(
    X_train,
    y_train
)

et_pred = et_model.predict(X_test)

et_results = evaluate_model(
    "Extra Trees",
    y_test,
    et_pred
)


Extra Trees
Accuracy: 0.7253
Macro Precision: 0.6816
Macro Recall: 0.6643
Macro F1: 0.6720

Classification Report:
              precision    recall  f1-score   support

  deforested       0.57      0.52      0.55        23
      forest       0.76      0.81      0.79       100
old_clearing       0.71      0.66      0.68        59

    accuracy                           0.73       182
   macro avg       0.68      0.66      0.67       182
weighted avg       0.72      0.73      0.72       182



In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_xgb = label_encoder.fit_transform(y_train)
y_test_xgb = label_encoder.transform(y_test)

print("Class mapping:")

for i, class_name in enumerate(label_encoder.classes_):
    print(i, "=", class_name)

Class mapping:
0 = deforested
1 = forest
2 = old_clearing


In [10]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_model.fit(
    X_train,
    y_train_xgb
)

xgb_pred_encoded = xgb_model.predict(X_test)

In [11]:
xgb_pred = label_encoder.inverse_transform(
    xgb_pred_encoded.astype(int)
)

xgb_results = evaluate_model(
    "XGBoost",
    y_test,
    xgb_pred
)


XGBoost
Accuracy: 0.7033
Macro Precision: 0.6519
Macro Recall: 0.6286
Macro F1: 0.6385

Classification Report:
              precision    recall  f1-score   support

  deforested       0.53      0.43      0.48        23
      forest       0.75      0.79      0.77       100
old_clearing       0.68      0.66      0.67        59

    accuracy                           0.70       182
   macro avg       0.65      0.63      0.64       182
weighted avg       0.70      0.70      0.70       182



In [12]:
results = pd.DataFrame([
    rf_results,
    et_results,
    xgb_results
])

results

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Random Forest,0.736264,0.705090,0.670919,0.685317
1,Extra Trees,0.725275,0.681557,0.664252,0.672024
2,XGBoost,0.703297,0.651936,0.628600,0.638532


In [14]:
results.to_csv(
    "../results/model_comparison.csv",
    index=False
)

print("Model comparison saved.")

Model comparison saved.


In [15]:
rf_cm = confusion_matrix(
    y_test,
    rf_pred,
    labels=[
        "deforested",
        "forest",
        "old_clearing"
    ]
)

rf_cm_df = pd.DataFrame(
    rf_cm,
    index=[
        "Actual deforested",
        "Actual forest",
        "Actual old_clearing"
    ],
    columns=[
        "Pred deforested",
        "Pred forest",
        "Pred old_clearing"
    ]
)

rf_cm_df

,Pred deforested,Pred forest,Pred old_clearing
Actual deforested,12,9,2
Actual forest,4,83,13
Actual old_clearing,3,17,39


In [17]:
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance

,Feature,Importance
3,ndvi_max,0.110802
5,ndvi_first,0.094835
6,ndvi_last,0.089326
7,overall_change,0.088894
1,ndvi_std,0.084440
10,early_mean,0.083357
0,ndvi_mean,0.075208
9,ndvi_slope,0.074335
12,early_late_change,0.059090
11,late_mean,0.054507


In [18]:
import sys
sys.path.append("../src")

from preprocessing import load_data, get_ndvi_columns, prepare_labels
from features import create_features
from models import create_random_forest, create_extra_trees
from evaluation import evaluate_model, get_confusion_matrix

print("All src imports successful!")

All src imports successful!


In [19]:
baseline_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Extra Trees",
        "XGBoost",
        "MLP",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        0.7362637363,
        0.7252747253,
        0.7032967033,
        0.6978021978,
        0.7087912088
    ],
    "Macro Precision": [
        0.7050896865,
        0.6815568080,
        0.6720318449,
        0.6686243461,
        0.6579752326
    ],
    "Macro Recall": [
        0.6709186932,
        0.6642520265,
        0.6374428887,
        0.5978924097,
        0.7012060919
    ],
    "Macro F1": [
        0.6853174771,
        0.6720242796,
        0.6516809346,
        0.6200966301,
        0.6726052419
    ]
})

baseline_results.to_csv(
    "../results/baseline_model_results.csv",
    index=False
)

print("Baseline results saved successfully.")
print(baseline_results)

Baseline results saved successfully.
                 Model  Accuracy  Macro Precision  Macro Recall  Macro F1
0        Random Forest  0.736264         0.705090      0.670919  0.685317
1          Extra Trees  0.725275         0.681557      0.664252  0.672024
2              XGBoost  0.703297         0.672032      0.637443  0.651681
3                  MLP  0.697802         0.668624      0.597892  0.620097
4  Tuned Random Forest  0.708791         0.657975      0.701206  0.672605


In [20]:
import pandas as pd

all_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Extra Trees",
        "XGBoost",
        "MLP",
        "Tuned Random Forest",
        "CNN"
    ],
    "Accuracy": [
        0.7362637363,
        0.7252747253,
        0.7032967033,
        0.6978021978,
        0.7087912088,
        0.6923076923
    ],
    "Macro Precision": [
        0.7050896865,
        0.6815568080,
        0.6720318449,
        0.6686243461,
        0.6579752326,
        0.6313932981
    ],
    "Macro Recall": [
        0.6709186932,
        0.6642520265,
        0.6374428887,
        0.5978924097,
        0.7012060919,
        0.5191820192
    ],
    "Macro F1": [
        0.6853174771,
        0.6720242796,
        0.6516809346,
        0.6200966301,
        0.6726052419,
        0.5074926254
    ]
})

all_results.to_csv(
    "../results/model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")
print(all_results.round(3))

Model comparison saved successfully.
                 Model  Accuracy  Macro Precision  Macro Recall  Macro F1
0        Random Forest     0.736            0.705         0.671     0.685
1          Extra Trees     0.725            0.682         0.664     0.672
2              XGBoost     0.703            0.672         0.637     0.652
3                  MLP     0.698            0.669         0.598     0.620
4  Tuned Random Forest     0.709            0.658         0.701     0.673
5                  CNN     0.692            0.631         0.519     0.507
